# 03 — Build Gold Analytics Model

Project: Databricks SQL Retail Analytics Dashboard  
Layer: Gold  
Source Table: retail_analytics_project.silver_retail_sales_clean  

Target Tables:

- retail_analytics_project.fact_sales
- retail_analytics_project.dim_product
- retail_analytics_project.dim_customer
- retail_analytics_project.dim_date
- retail_analytics_project.gold_data_quality_summary

This notebook builds a dimensional analytics model from the cleaned Silver retail transactions.

The Gold model follows a star-schema approach:

- `fact_sales` stores commercial transaction-line events.
- `dim_product` provides standardized product attributes.
- `dim_customer` provides customer-level context.
- `dim_date` supports time-based analytics.
- `gold_data_quality_summary` exposes important data quality and operational exceptions.

In [0]:
from pyspark.sql import Window

from pyspark.sql.functions import (
    col,
    when,
    lit,
    abs as spark_abs,
    round as spark_round,
    row_number,
    count,
    min as spark_min,
    max as spark_max,
    sum as spark_sum,
    desc,
    concat_ws,
    lpad,
    to_date,
    year,
    month,
    quarter,
    dayofmonth,
    dayofweek,
    weekofyear,
    date_format,
    current_timestamp,
    sequence,
    explode
)

In [0]:
schema_name = "retail_analytics_project"

silver_table = f"{schema_name}.silver_retail_sales_clean"

fact_sales_table = f"{schema_name}.fact_sales"
dim_product_table = f"{schema_name}.dim_product"
dim_customer_table = f"{schema_name}.dim_customer"
dim_date_table = f"{schema_name}.dim_date"
quality_summary_table = f"{schema_name}.gold_data_quality_summary"

print(f"Silver source: {silver_table}")
print(f"Fact table: {fact_sales_table}")
print(f"Product dimension: {dim_product_table}")
print(f"Customer dimension: {dim_customer_table}")
print(f"Date dimension: {dim_date_table}")

In [0]:
if not spark.catalog.tableExists(silver_table):
    raise Exception(f"Required Silver table does not exist: {silver_table}")

silver_df = spark.table(silver_table)

silver_count = silver_df.count()

print(f"Silver table found: {silver_table}")
print(f"Silver records available: {silver_count}")

In [0]:
commercial_events_df = (
    silver_df
    .filter(
        col("transaction_type").isin(
            "SALE",
            "CANCELLATION"
        )
    )
)

In [0]:
display(
    commercial_events_df
    .groupBy("transaction_type")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
invoice_line_window = (
    Window
    .partitionBy("invoice_no")
    .orderBy(
        col("stock_code"),
        col("product_description"),
        col("quantity"),
        col("unit_price"),
        col("customer_id")
    )
)

commercial_events_df = (
    commercial_events_df
    .withColumn(
        "invoice_line_number",
        row_number().over(invoice_line_window)
    )
)

In [0]:
commercial_events_df = (
    commercial_events_df
    .withColumn(
        "sales_line_id",
        concat_ws(
            "-",
            col("invoice_no"),
            lpad(
                col("invoice_line_number").cast("string"),
                4,
                "0"
            )
        )
    )
)

In [0]:
fact_sales_df = (
    commercial_events_df
    .withColumn(
        "sold_quantity",
        when(
            col("transaction_type") == "SALE",
            col("quantity")
        ).otherwise(lit(0))
    )
    .withColumn(
        "cancelled_quantity",
        when(
            col("transaction_type") == "CANCELLATION",
            spark_abs(col("quantity"))
        ).otherwise(lit(0))
    )
    .withColumn(
        "net_quantity",
        col("sold_quantity") - col("cancelled_quantity")
    )
    .withColumn(
        "net_sales_amount",
        spark_round(
            col("gross_sales_amount")
            - col("cancellation_amount"),
            2
        )
    )
)

In [0]:
fact_sales_df = fact_sales_df.select(
    "sales_line_id",
    "invoice_no",
    "invoice_line_number",
    "invoice_date",
    "invoice_ts",
    "stock_code",
    "customer_id",
    "country",
    "transaction_type",
    "quantity",
    "sold_quantity",
    "cancelled_quantity",
    "net_quantity",
    "unit_price",
    "gross_sales_amount",
    "cancellation_amount",
    "net_sales_amount",
    "is_customer_known",
    "source_system",
    "bronze_ingestion_ts",
    "silver_processed_ts",
    current_timestamp().alias("gold_processed_ts")
)

display(fact_sales_df.limit(20))

In [0]:
display(
    fact_sales_df.agg(
        count("*").alias("fact_rows"),
        spark_round(
            spark_sum("gross_sales_amount"),
            2
        ).alias("gross_sales"),
        spark_round(
            spark_sum("cancellation_amount"),
            2
        ).alias("cancellation_amount"),
        spark_round(
            spark_sum("net_sales_amount"),
            2
        ).alias("net_sales"),
        spark_sum("sold_quantity").alias("sold_units"),
        spark_sum("cancelled_quantity").alias("cancelled_units"),
        spark_sum("net_quantity").alias("net_units")
    )
)

In [0]:
fact_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(fact_sales_table)

print(f"Fact table created successfully: {fact_sales_table}")

In [0]:
product_description_counts_df = (
    silver_df
    .filter(
        col("stock_code").isNotNull()
        & (col("stock_code") != "")
        & (col("product_description") != "UNKNOWN PRODUCT")
    )
    .groupBy(
        "stock_code",
        "product_description"
    )
    .agg(
        count("*").alias("description_usage_count")
    )
)

In [0]:
product_description_window = (
    Window
    .partitionBy("stock_code")
    .orderBy(
        desc("description_usage_count"),
        col("product_description")
    )
)

dim_product_df = (
    product_description_counts_df
    .withColumn(
        "description_rank",
        row_number().over(product_description_window)
    )
    .filter(col("description_rank") == 1)
    .select(
        "stock_code",
        col("product_description").alias(
            "canonical_product_description"
        ),
        "description_usage_count"
    )
)

In [0]:
product_metrics_df = (
    silver_df
    .filter(col("stock_code").isNotNull())
    .groupBy("stock_code")
    .agg(
        count("*").alias("total_source_rows"),
        spark_min("invoice_date").alias("first_seen_date"),
        spark_max("invoice_date").alias("last_seen_date"),
        spark_round(
            spark_sum("gross_sales_amount"),
            2
        ).alias("lifetime_gross_sales"),
        spark_round(
            spark_sum("cancellation_amount"),
            2
        ).alias("lifetime_cancellation_amount")
    )
)

dim_product_df = (
    dim_product_df
    .join(
        product_metrics_df,
        on="stock_code",
        how="left"
    )
    .withColumn(
        "is_non_product_code",
        col("stock_code").isin(
            "POST",
            "D",
            "DOT",
            "M",
            "AMAZONFEE",
            "BANK CHARGES",
            "B"
        )
    )
    .withColumn(
        "gold_processed_ts",
        current_timestamp()
    )
)

In [0]:
dim_product_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(dim_product_table)

print(f"Product dimension created: {dim_product_table}")

In [0]:
customer_country_counts_df = (
    silver_df
    .groupBy(
        "customer_id",
        "country"
    )
    .agg(
        count("*").alias("country_usage_count")
    )
)

customer_country_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        desc("country_usage_count"),
        col("country")
    )
)

customer_main_country_df = (
    customer_country_counts_df
    .withColumn(
        "country_rank",
        row_number().over(customer_country_window)
    )
    .filter(col("country_rank") == 1)
    .select(
        "customer_id",
        col("country").alias("primary_country")
    )
)

In [0]:
customer_metrics_df = (
    silver_df
    .groupBy("customer_id")
    .agg(
        spark_min("invoice_date").alias("first_activity_date"),
        spark_max("invoice_date").alias("last_activity_date"),
        count("*").alias("total_transaction_lines"),
        count("invoice_no").alias("invoice_line_count"),
        spark_round(
            spark_sum("gross_sales_amount"),
            2
        ).alias("lifetime_gross_sales"),
        spark_round(
            spark_sum("cancellation_amount"),
            2
        ).alias("lifetime_cancellation_amount"),
        spark_round(
            spark_sum(
                col("gross_sales_amount")
                - col("cancellation_amount")
            ),
            2
        ).alias("lifetime_net_sales")
    )
)

dim_customer_df = (
    customer_main_country_df
    .join(
        customer_metrics_df,
        on="customer_id",
        how="left"
    )
    .withColumn(
        "is_known_customer",
        col("customer_id") != "UNKNOWN"
    )
    .withColumn(
        "gold_processed_ts",
        current_timestamp()
    )
)

In [0]:
dim_customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(dim_customer_table)

print(f"Customer dimension created: {dim_customer_table}")

In [0]:
date_range = (
    silver_df
    .agg(
        spark_min("invoice_date").alias("min_date"),
        spark_max("invoice_date").alias("max_date")
    )
    .collect()[0]
)

min_date = date_range["min_date"]
max_date = date_range["max_date"]

print(f"Minimum date: {min_date}")
print(f"Maximum date: {max_date}")

In [0]:
dim_date_df = (
    spark
    .range(1)
    .select(
        explode(
            sequence(
                lit(min_date),
                lit(max_date)
            )
        ).alias("date")
    )
    .withColumn(
        "date_key",
        date_format(col("date"), "yyyyMMdd").cast("int")
    )
    .withColumn("year", year(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn(
        "month_name",
        date_format(col("date"), "MMMM")
    )
    .withColumn(
        "year_month",
        date_format(col("date"), "yyyy-MM")
    )
    .withColumn(
        "day_of_month",
        dayofmonth(col("date"))
    )
    .withColumn(
        "day_of_week_number",
        dayofweek(col("date"))
    )
    .withColumn(
        "day_name",
        date_format(col("date"), "EEEE")
    )
    .withColumn(
        "week_of_year",
        weekofyear(col("date"))
    )
    .withColumn(
        "is_weekend",
        dayofweek(col("date")).isin(1, 7)
    )
    .withColumn(
        "gold_processed_ts",
        current_timestamp()
    )
)

In [0]:
dim_date_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(dim_date_table)

print(f"Date dimension created: {dim_date_table}")

In [0]:
quality_summary_df = spark.sql(f"""
SELECT
    COUNT(*) AS total_silver_rows,

    SUM(
        CASE
            WHEN customer_id = 'UNKNOWN'
            THEN 1 ELSE 0
        END
    ) AS unknown_customer_rows,

    SUM(
        CASE
            WHEN product_description = 'UNKNOWN PRODUCT'
            THEN 1 ELSE 0
        END
    ) AS unknown_product_rows,

    SUM(
        CASE
            WHEN transaction_type = 'SALE'
            THEN 1 ELSE 0
        END
    ) AS sale_rows,

    SUM(
        CASE
            WHEN transaction_type = 'CANCELLATION'
            THEN 1 ELSE 0
        END
    ) AS cancellation_rows,

    SUM(
        CASE
            WHEN transaction_type = 'RETURN_OR_ADJUSTMENT'
            THEN 1 ELSE 0
        END
    ) AS inventory_adjustment_rows,

    SUM(
        CASE
            WHEN transaction_type = 'INVALID_OR_NON_COMMERCIAL'
            THEN 1 ELSE 0
        END
    ) AS non_commercial_rows,

    ROUND(
        ABS(
            SUM(
                CASE
                    WHEN stock_code = 'B'
                    THEN signed_line_amount
                    ELSE 0
                END
            )
        ),
        2
    ) AS bad_debt_net_loss_amount,

    current_timestamp() AS gold_processed_ts

FROM {silver_table}
""")

In [0]:
quality_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(quality_summary_table)

print(f"Quality summary created: {quality_summary_table}")

In [0]:
display(
    spark.sql(f"""
    SELECT 'fact_sales' AS table_name, COUNT(*) AS total_records
    FROM {fact_sales_table}

    UNION ALL

    SELECT 'dim_product', COUNT(*)
    FROM {dim_product_table}

    UNION ALL

    SELECT 'dim_customer', COUNT(*)
    FROM {dim_customer_table}

    UNION ALL

    SELECT 'dim_date', COUNT(*)
    FROM {dim_date_table}

    UNION ALL

    SELECT 'gold_data_quality_summary', COUNT(*)
    FROM {quality_summary_table}
    """)
)

In [0]:
display(
    spark.sql(f"""
    SELECT
        SUM(
            CASE WHEN p.stock_code IS NULL
            THEN 1 ELSE 0 END
        ) AS fact_rows_without_product,

        SUM(
            CASE WHEN c.customer_id IS NULL
            THEN 1 ELSE 0 END
        ) AS fact_rows_without_customer,

        SUM(
            CASE WHEN d.date IS NULL
            THEN 1 ELSE 0 END
        ) AS fact_rows_without_date

    FROM {fact_sales_table} f

    LEFT JOIN {dim_product_table} p
        ON f.stock_code = p.stock_code

    LEFT JOIN {dim_customer_table} c
        ON f.customer_id = c.customer_id

    LEFT JOIN {dim_date_table} d
        ON f.invoice_date = d.date
    """)
)